# 7. Ontology-Checked Graph Queries: Text-to-SPARQL with Deterministic Checks, Repair and a Read-Only Gate

**Presentation map:** Slide 27 (RDF · OWL · SPARQL — formal semantics a checker can enforce), Slide 43 (exposing a graph
to an agent as a query tool), Slide 44 (free-form text-to-query is unreliable; ground entities, project the schema,
refine deterministically), Slide 59 (an ontology layer raised enterprise QA accuracy from 16% to 72.55%),
Slide 60 (ontology-based query check: deterministic rules, LLM repair, "unknown" after 3 failures),
Slide 61 (guardrails outside the prompt: read-only by query plan), Slide 50 (structural validity is not factual truth).

**Dataset:** `data/aura_helpdesk.json` — the same Aura IT-helpdesk graph as Notebooks 1 and 5 (23 tickets, 57 comments,
10 users, 10 KB articles, 246 relationships), converted here into **RDF triples** with a small OWL-style ontology
(`rdfs:domain` / `rdfs:range` on every property).

**The experiment (after Sequeda & Allemang, arXiv 2405.11706):** 15 natural-language questions with gold answers
computed by hand-written SPARQL. The LLM writes SPARQL under two conditions:

- **(a) Unchecked** — the LLM sees only a terse list of class and property names (what most "chat with your graph"
  demos give it) and its query is executed as-is.
- **(b) Ontology-checked** — the same prompt, but every query passes a **deterministic checker** before execution:
  read-only gate, unknown predicates, domain/range and direction violations, literal datatypes, and entity grounding
  of string constants. Violations are explained back to the LLM, which may **repair at most 3 times**; otherwise the
  system answers **"unknown"** instead of guessing.

In [1]:
import json
from collections import Counter
from rdflib import Graph, Namespace, Literal, RDF, RDFS, OWL, XSD

with open("data/aura_helpdesk.json") as f:
    data = json.load(f)

EX = Namespace("http://example.org/aura#")
g = Graph()
g.bind("ex", EX)

# ---------------- Ontology (TBox): classes + properties with domain / range ----------------
CLASSES = ["Ticket", "Comment", "User", "Status", "Category", "Priority", "Documentation"]
OBJECT_PROPS = {            # property: (domain, range) — directions taken verbatim from the Neo4j load script
    "submits":          ("User", "Ticket"),          # (User)-[:SUBMITS]->(Ticket)
    "assignedTo":       ("User", "Ticket"),          # (User)-[:ASSIGNED_TO]->(Ticket)   <- direction is a classic trap
    "contains":         ("Ticket", "Comment"),       # (Ticket)-[:CONTAINS]->(Comment)
    "posts":            ("User", "Comment"),         # (User)-[:POSTS]->(Comment)
    "hasStatus":        ("Ticket", "Status"),
    "hasPriority":      ("Ticket", "Priority"),
    "hasCategory":      ("Ticket", "Category"),
    "hasDocumentation": ("Category", "Documentation"),  # (Category)-[:HAS_CATEGORY]->(Documentation) in the source
    "referencedBy":     ("Documentation", "Comment"),   # (Documentation)-[:REFERENCED_BY]->(Comment)
}
DATA_PROPS = {              # property: (domain(s), xsd range)
    "title":        ({"Ticket", "Documentation"}, XSD.string),
    "description":  ({"Ticket"}, XSD.string),
    "content":      ({"Comment"}, XSD.string),
    "name":         ({"Status", "Category", "Priority"}, XSD.string),
    "firstName":    ({"User"}, XSD.string),
    "lastName":     ({"User"}, XSD.string),
    "role":         ({"User"}, XSD.string),
    "isSolution":   ({"Comment"}, XSD.boolean),
    "isClosed":     ({"Status"}, XSD.boolean),
    "level":        ({"Priority"}, XSD.integer),
    "creationDate": ({"Ticket", "Comment"}, XSD.dateTime),
}
for c in CLASSES:
    g.add((EX[c], RDF.type, OWL.Class))
for p, (d, r) in OBJECT_PROPS.items():
    g.add((EX[p], RDF.type, OWL.ObjectProperty)); g.add((EX[p], RDFS.domain, EX[d])); g.add((EX[p], RDFS.range, EX[r]))
for p, (ds, r) in DATA_PROPS.items():
    g.add((EX[p], RDF.type, OWL.DatatypeProperty)); g.add((EX[p], RDFS.range, r))
    for d in ds:
        g.add((EX[p], RDFS.domain, EX[d]))

# ---------------- Instance data (ABox) ----------------
def lit(v, dt=XSD.string):
    if dt == XSD.boolean: return Literal(str(v).lower() == "true", datatype=XSD.boolean)
    if dt == XSD.integer: return Literal(int(v), datatype=XSD.integer)
    if dt == XSD.dateTime: return Literal(str(v).replace(" ", "T"), datatype=XSD.dateTime)
    return Literal(v)

def add_entity(uid, cls, fields):
    g.add((EX[uid], RDF.type, EX[cls]))
    for prop, value in fields.items():
        if value not in ("", None):
            g.add((EX[uid], EX[prop], lit(value, DATA_PROPS[prop][1])))

for x in data["statuses"]:   add_entity(x["statusId"], "Status", {"name": x["name"], "isClosed": x["isClosed"]})
for x in data["priorities"]: add_entity(x["priorityId"], "Priority", {"name": x["name"], "level": x["level"]})
for x in data["categories"]: add_entity(x["categoryId"], "Category", {"name": x["name"]})
for x in data["users"]:      add_entity(x["userId"], "User", {"firstName": x["firstName"], "lastName": x["lastName"], "role": x["role"]})
for x in data["tickets"]:    add_entity(x["ticketId"], "Ticket", {"title": x["title"], "description": x["description"], "creationDate": x["creationDate"]})
for x in data["comments"]:   add_entity(x["commentId"], "Comment", {"content": x["content"], "isSolution": x["isSolution"], "creationDate": x["creationDate"]})
for x in data["documentation"]: add_entity(x["articleId"], "Documentation", {"title": x["title"]})

REL_TO_PROP = {("SUBMITS", "User"): "submits", ("ASSIGNED_TO", "User"): "assignedTo", ("CONTAINS", "Ticket"): "contains",
               ("POSTS", "User"): "posts", ("HAS_STATUS", "Ticket"): "hasStatus", ("HAS_PRIORITY", "Ticket"): "hasPriority",
               ("HAS_CATEGORY", "Ticket"): "hasCategory", ("HAS_CATEGORY", "Category"): "hasDocumentation",
               ("REFERENCED_BY", "DocumentationArticle"): "referencedBy"}
for r in data["relationships"]:
    g.add((EX[r["from"]], EX[REL_TO_PROP[(r["type"], r["from_label"])]], EX[r["to"]]))

dangling = sorted({str(o).split("#")[-1] for o in g.objects(None, EX.hasCategory) if (o, RDF.type, EX.Category) not in g})
print(f"RDF graph: {len(g)} triples")
print(f"Data-quality note: ex:hasCategory points to undefined categories {dangling} (tkt_011 -> cat_010 is not in the source categories)")
print("Instances per class:", {c: len(set(g.subjects(RDF.type, EX[c]))) for c in CLASSES})
print("Object-property triples:", {p: len(list(g.triples((None, EX[p], None)))) for p in OBJECT_PROPS})

RDF graph: 751 triples
Data-quality note: ex:hasCategory points to undefined categories ['cat_010'] (tkt_011 -> cat_010 is not in the source categories)
Instances per class: {'Ticket': 23, 'Comment': 57, 'User': 10, 'Status': 7, 'Category': 9, 'Priority': 5, 'Documentation': 10}
Object-property triples: {'submits': 23, 'assignedTo': 23, 'contains': 57, 'posts': 57, 'hasStatus': 23, 'hasPriority': 23, 'hasCategory': 23, 'hasDocumentation': 10, 'referencedBy': 7}


## The benchmark: 15 questions, gold answers from hand-written SPARQL

Each gold answer is a list of rows; a cell in a row matches either the entity's id (e.g. `tkt_001`) **or** its
human-readable label, so the LLM is not penalised for returning ids instead of titles. A predicted answer is correct
when it has the same number of distinct rows and every gold row is matched.

In [2]:
PFX = "PREFIX ex: <http://example.org/aura#>\nPREFIX xsd: <http://www.w3.org/2001/XMLSchema#>\n"

QUESTIONS = [
    ("Q01", "Which tickets were submitted by Marie Dubois? Return the ticket titles.",
     "SELECT ?t ?title WHERE { ?u ex:firstName 'Marie' ; ex:lastName 'Dubois' ; ex:submits ?t . ?t ex:title ?title }"),
    ("Q02", "Which tickets is Jean Martin assigned to? Return the ticket titles.",
     "SELECT ?t ?title WHERE { ?u ex:firstName 'Jean' ; ex:lastName 'Martin' ; ex:assignedTo ?t . ?t ex:title ?title }"),
    ("Q03", "How many tickets have the status 'Resolved'?",
     "SELECT (COUNT(DISTINCT ?t) AS ?n) WHERE { ?t ex:hasStatus ?s . ?s ex:name 'Resolved' }"),
    ("Q04", "List the titles of tickets with priority 'High'.",
     "SELECT ?t ?title WHERE { ?t ex:hasPriority ?p . ?p ex:name 'High' . ?t ex:title ?title }"),
    ("Q05", "Who (first and last name) posted the comment marked as the solution on the ticket 'Cannot login to Aura'?",
     "SELECT ?u ?fn ?ln WHERE { ?t ex:title 'Cannot login to Aura' ; ex:contains ?c . ?c ex:isSolution true . ?u ex:posts ?c ; ex:firstName ?fn ; ex:lastName ?ln }"),
    ("Q06", "Which documentation articles are referenced by comments on the ticket 'Cannot login to Aura'? Return article titles.",
     "SELECT DISTINCT ?d ?title WHERE { ?t ex:title 'Cannot login to Aura' ; ex:contains ?c . ?d ex:referencedBy ?c ; ex:title ?title }"),
    ("Q07", "Which users with the role 'agent' have posted at least one comment marked as a solution? Return first names.",
     "SELECT DISTINCT ?u ?fn WHERE { ?u ex:role 'agent' ; ex:posts ?c ; ex:firstName ?fn . ?c ex:isSolution true }"),
    ("Q08", "Which tickets in the category 'Aura Access' have a status that is not closed? Return titles.",
     "SELECT ?t ?title WHERE { ?t ex:hasCategory ?c . ?c ex:name 'Aura Access' . ?t ex:hasStatus ?s . ?s ex:isClosed false . ?t ex:title ?title }"),
    ("Q09", "How many comments has each user with the role 'manager' posted? Return first name and count.",
     "SELECT ?u ?fn (COUNT(?c) AS ?n) WHERE { ?u ex:role 'manager' ; ex:firstName ?fn ; ex:posts ?c } GROUP BY ?u ?fn"),
    ("Q10", "Which documentation articles belong to the category of the ticket 'Cannot login to Aura'? Return article titles.",
     "SELECT ?d ?title WHERE { ?t ex:title 'Cannot login to Aura' ; ex:hasCategory ?c . ?c ex:hasDocumentation ?d . ?d ex:title ?title }"),
    ("Q11", "Who submitted the tickets that Anna Kowalski is assigned to? Return distinct submitter first names.",
     "SELECT DISTINCT ?s ?fn WHERE { ?a ex:firstName 'Anna' ; ex:lastName 'Kowalski' ; ex:assignedTo ?t . ?s ex:submits ?t ; ex:firstName ?fn }"),
    ("Q12", "How many tickets have no comment marked as a solution?",
     "SELECT (COUNT(DISTINCT ?t) AS ?n) WHERE { ?t a ex:Ticket . FILTER NOT EXISTS { ?t ex:contains ?c . ?c ex:isSolution true } }"),
    ("Q13", "Which tickets have at least one comment posted by a user with the role 'manager'? Return titles.",
     "SELECT DISTINCT ?t ?title WHERE { ?t ex:contains ?c ; ex:title ?title . ?u ex:posts ?c ; ex:role 'manager' }"),
    ("Q14", "Which priority level name has the most tickets, and how many tickets does it have?",
     "SELECT ?p ?name (COUNT(?t) AS ?n) WHERE { ?t ex:hasPriority ?p . ?p ex:name ?name } GROUP BY ?p ?name ORDER BY DESC(?n) LIMIT 1"),
    ("Q15", "Which employees (role 'employee') submitted a ticket in the category 'Aura Responses'? Return distinct first names.",
     "SELECT DISTINCT ?u ?fn WHERE { ?u ex:role 'employee' ; ex:submits ?t ; ex:firstName ?fn . ?t ex:hasCategory ?c . ?c ex:name 'Aura Responses' }"),
]

def short(v):
    s = str(v)
    return s.split("#")[-1] if s.startswith(str(EX)) else s

def gold_rows(sparql):
    """Gold rows: hand-written queries return (id, label...) - each id/label pair becomes one alternatives-set."""
    rows = []
    for r in g.query(PFX + sparql):
        vals = [short(v) for v in r]
        if len(vals) >= 2 and vals[0].split("_")[0] in {"tkt", "usr", "doc", "pri", "cat", "sts", "cmt"}:
            rows.append([{vals[0], vals[1]}] + [{v} for v in vals[2:]])
        else:
            rows.append([{v} for v in vals])
    return rows

GOLD = {qid: gold_rows(q) for qid, _, q in QUESTIONS}
for qid, text, _ in QUESTIONS:
    print(f"{qid}  {len(GOLD[qid]):>2} gold rows  {text}")

Q01   2 gold rows  Which tickets were submitted by Marie Dubois? Return the ticket titles.
Q02   4 gold rows  Which tickets is Jean Martin assigned to? Return the ticket titles.
Q03   1 gold rows  How many tickets have the status 'Resolved'?
Q04   8 gold rows  List the titles of tickets with priority 'High'.
Q05   1 gold rows  Who (first and last name) posted the comment marked as the solution on the ticket 'Cannot login to Aura'?
Q06   1 gold rows  Which documentation articles are referenced by comments on the ticket 'Cannot login to Aura'? Return article titles.
Q07   2 gold rows  Which users with the role 'agent' have posted at least one comment marked as a solution? Return first names.
Q08   1 gold rows  Which tickets in the category 'Aura Access' have a status that is not closed? Return titles.
Q09   2 gold rows  How many comments has each user with the role 'manager' posted? Return first name and count.
Q10   1 gold rows  Which documentation articles belong to the category of the

## The deterministic ontology checker (Slides 60-61)

The checker never calls an LLM. It parses the query with rdflib's SPARQL parser, walks the algebra, and checks:

1. **Read-only gate** — anything that parses as a SPARQL *Update* (`INSERT`, `DELETE`, `DROP`, ...) is rejected
   before it reaches the store, and only `SELECT`/`ASK` are allowed: *permission by query plan, not by prompt*.
2. **Vocabulary** — every `ex:` predicate and class must exist in the ontology.
3. **Domain / range / direction** — every variable gets the classes implied by its *other* patterns
   (`?u ex:firstName 'Jean'` ⇒ `?u` is a User). If `?t ex:assignedTo ?u` then puts a User in the object slot, the
   property is reversed (`ex:assignedTo` goes User → Ticket) and the checker says exactly how to swap it.
4. **Literal datatypes** — `ex:isSolution "true"` (a string) is flagged: the range is `xsd:boolean`.
5. **Entity grounding** — a string constant compared with `ex:name`, `ex:title`, `ex:firstName`, ... must exist in
   the graph; otherwise the closest existing values are suggested (Slide 44: *ground entities first*).

In [3]:
import difflib, re
from rdflib.plugins.sparql import prepareQuery
from rdflib.plugins.sparql.parser import parseUpdate
from rdflib.term import Variable, URIRef

LABEL_PROPS = {"name", "title", "firstName", "lastName", "role"}
KNOWN_VALUES = {p: sorted({str(o) for o in g.objects(None, EX[p])}) for p in LABEL_PROPS}

def iter_triples(node):
    """Yield every triple pattern in an rdflib SPARQL algebra tree (BGPs inside joins, filters, NOT EXISTS, ...)."""
    if isinstance(node, dict):
        if getattr(node, "name", None) == "BGP":
            yield from node["triples"]
        for v in node.values():
            yield from iter_triples(v)
    elif isinstance(node, (list, tuple, set)):
        for v in node:
            yield from iter_triples(v)

def local(u):
    return str(u).split("#")[-1] if isinstance(u, URIRef) and str(u).startswith(str(EX)) else None

def check_query(q: str):
    """Return (ok, parsed_query_or_None, [violations])."""
    q_full = q if "PREFIX ex:" in q else PFX + q
    # --- 1. read-only gate -------------------------------------------------------------------
    try:
        parsed = prepareQuery(q_full)
    except Exception as e:
        try:
            parseUpdate(q_full)
            is_write = True
        except Exception:      # not valid SPARQL Update either - but still refuse anything that *looks* like a write
            is_write = re.match(r"\s*(INSERT|DELETE|UPDATE|DROP|CLEAR|LOAD|CREATE|MODIFY|WITH|ADD|MOVE|COPY)\b", q, re.I) is not None
        if is_write:
            return False, None, ["READ-ONLY GATE: this is a write operation (INSERT/DELETE/UPDATE/...). Only SELECT queries are permitted."]
        return False, None, [f"SYNTAX: query does not parse ({str(e)[:160]})"]
    if parsed.algebra.name not in ("SelectQuery", "AskQuery"):
        return False, None, [f"READ-ONLY GATE: {parsed.algebra.name} is not permitted; write a SELECT query."]

    violations, implied, declared = [], {}, {}
    triples = list(iter_triples(parsed.algebra))
    for i, (s, p, o) in enumerate(triples):
        if p == RDF.type and local(o) is not None:
            if local(o) not in CLASSES:
                violations.append(f"VOCABULARY: class ex:{local(o)} does not exist. Classes: {', '.join(CLASSES)}.")
            elif isinstance(s, Variable):
                declared.setdefault(s, set()).add(local(o))
            continue
        name = local(p)
        if name is None:
            continue
        # --- 2. vocabulary ------------------------------------------------------------------------
        if name not in OBJECT_PROPS and name not in DATA_PROPS:
            close = difflib.get_close_matches(name, list(OBJECT_PROPS) + list(DATA_PROPS), n=2)
            violations.append(f"VOCABULARY: property ex:{name} does not exist" + (f" (did you mean {', '.join('ex:'+c for c in close)}?)" if close else "."))
            continue
        if name in OBJECT_PROPS:
            dom, rng = OBJECT_PROPS[name]
            for term, cls in ((s, dom), (o, rng)):          # constant IRIs must be real entities of the right class
                if local(term) is not None and (term, RDF.type, EX[cls]) not in g:
                    examples = [f"?x ex:{lp} \"{v}\"" for lp in sorted(LABEL_PROPS) for v in KNOWN_VALUES[lp]
                                if difflib.SequenceMatcher(None, local(term).lower(), v.replace(" ", "").lower()).ratio() > 0.6][:2]
                    violations.append(f"GROUNDING: ex:{local(term)} is not a {cls} in the graph. Match entities by their "
                                      f"attributes instead" + (f", e.g. {' or '.join(examples)}." if examples else "."))
            if isinstance(s, Variable): implied.setdefault(s, []).append(({dom}, i))
            if isinstance(o, Variable): implied.setdefault(o, []).append(({rng}, i))
            if isinstance(o, Literal):
                violations.append(f"RANGE: ex:{name} links a {dom} to a {rng} resource, not to a literal {o.n3()}.")
        else:
            doms, rng = DATA_PROPS[name]
            if isinstance(s, Variable): implied.setdefault(s, []).append((doms, i))
            # --- 4. literal datatypes -------------------------------------------------------------
            if isinstance(o, Literal):
                lit_dt = o.datatype or XSD.string
                if rng != XSD.string and lit_dt != rng:
                    violations.append(f"DATATYPE: ex:{name} has range {rng.n3(g.namespace_manager)}; {o.n3()} is a {lit_dt.n3(g.namespace_manager)}. "
                                      f"Write it as {'true/false' if rng == XSD.boolean else 'a typed literal'}.")
                # --- 5. entity grounding ----------------------------------------------------------
                if name in LABEL_PROPS and str(o) not in KNOWN_VALUES[name]:
                    elsewhere = [lp for lp in LABEL_PROPS if str(o) in KNOWN_VALUES[lp]]
                    if elsewhere:
                        owners = sorted({local(c) for x in g.subjects(EX[elsewhere[0]], Literal(str(o))) for c in g.objects(x, RDF.type)})
                        violations.append(f"GROUNDING: \"{o}\" is the ex:{elsewhere[0]} of a {'/'.join(owners)}, not an ex:{name} "
                                          f"(ex:{name} is only used on {'/'.join(sorted(doms))}).")
                    else:
                        close = difflib.get_close_matches(str(o), KNOWN_VALUES[name], n=3, cutoff=0.4)
                        violations.append(f"GROUNDING: no entity has ex:{name} \"{o}\". Closest existing values: {close}.")
    # --- 3. domain / range / direction: check each object-property triple against the classes that
    #        the OTHER patterns imply for its subject and object variables -----------------------
    def classes_elsewhere(var, skip):
        sets = [cls for cls, j in implied.get(var, []) if j != skip] + ([declared[var]] if var in declared else [])
        common = set.intersection(*map(set, sets)) if sets else None
        return common or None          # an empty set means the other uses already conflict; reported there
    for i, (s, p, o) in enumerate(triples):
        name = local(p)
        if name not in OBJECT_PROPS:
            continue
        dom, rng = OBJECT_PROPS[name]
        s_cls = classes_elsewhere(s, i) if isinstance(s, Variable) else None
        o_cls = classes_elsewhere(o, i) if isinstance(o, Variable) else None
        s_bad = s_cls is not None and dom not in s_cls
        o_bad = o_cls is not None and rng not in o_cls
        if not (s_bad or o_bad):
            continue
        if (s_cls is None or rng in s_cls) and (o_cls is None or dom in o_cls):
            violations.append(f"DIRECTION: `{s.n3()} ex:{name} {o.n3()}` is reversed. ex:{name} goes from a {dom} to a {rng}; "
                              f"write `{o.n3()} ex:{name} {s.n3()}`.")
        else:
            side = f"subject {s.n3()} is used elsewhere as a {'/'.join(sorted(s_cls))}" if s_bad else \
                   f"object {o.n3()} is used elsewhere as a {'/'.join(sorted(o_cls))}"
            violations.append(f"DOMAIN/RANGE: ex:{name} links a {dom} (subject) to a {rng} (object), but the {side}.")
    return (not violations), parsed, violations

# Sanity: every gold query passes the checker
assert all(check_query(q)[0] for _, _, q in QUESTIONS), [check_query(q)[2] for _, _, q in QUESTIONS if not check_query(q)[0]]
print("All 15 hand-written gold queries pass the checker.\n")

for bad in ["SELECT ?t WHERE { ?t ex:assignedTo ?u . ?u ex:firstName 'Jean' }",
            "SELECT (COUNT(?t) AS ?n) WHERE { ?t ex:hasStatus ex:Resolved }",
            "SELECT ?c WHERE { ?c ex:isSolution 'true' }",
            "SELECT ?t WHERE { ?t ex:hasStatus ?s . ?s ex:name 'Solved' }"]:
    print(bad, "\n   ->", check_query(bad)[2], "\n")

All 15 hand-written gold queries pass the checker.

SELECT ?t WHERE { ?t ex:assignedTo ?u . ?u ex:firstName 'Jean' } 
   -> ['DIRECTION: `?t ex:assignedTo ?u` is reversed. ex:assignedTo goes from a User to a Ticket; write `?u ex:assignedTo ?t`.'] 

SELECT (COUNT(?t) AS ?n) WHERE { ?t ex:hasStatus ex:Resolved } 
   -> ['GROUNDING: ex:Resolved is not a Status in the graph. Match entities by their attributes instead, e.g. ?x ex:name "Resolved".'] 

SELECT ?c WHERE { ?c ex:isSolution 'true' } 
   -> ['DATATYPE: ex:isSolution has range xsd:boolean; "true" is a xsd:string. Write it as true/false.'] 

SELECT ?t WHERE { ?t ex:hasStatus ?s . ?s ex:name 'Solved' } 
   -> ['GROUNDING: no entity has ex:name "Solved". Closest existing values: [\'Resolved\', \'Closed\', \'Cancelled\'].'] 



## Read-only gate: write attempts never reach the store (Slide 61)

An agent that has read a poisoned ticket (Notebook 5) might be steered into "fixing" data. The gate rejects updates
by **parsing** them — no instruction in the prompt is involved.

In [4]:
import os
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
CHAT_MODEL = "gpt-4o-mini"

# What most "chat with your graph" demos give the model: names only, no directions, no datatypes.
TERSE_SCHEMA = ("Prefix ex: <http://example.org/aura#>\n"
                "Classes: " + ", ".join(f"ex:{c}" for c in CLASSES) + "\n"
                "Relationships between entities: " + ", ".join(f"ex:{p}" for p in OBJECT_PROPS) + "\n"
                "Attributes: " + ", ".join(f"ex:{p}" for p in DATA_PROPS))

SYSTEM = ("You translate questions about an IT-helpdesk knowledge graph into a single SPARQL query.\n"
          f"Schema:\n{TERSE_SCHEMA}\n"
          "Return only the query inside one ```sparql code block. Do not include PREFIX lines.")

def llm_sparql(messages):
    out = client.chat.completions.create(model=CHAT_MODEL, temperature=0, seed=7, messages=messages).choices[0].message.content
    return out.split("```sparql")[-1].split("```")[0].strip() if "```" in out else out.strip()

# An instruction that arrives in the request (e.g. copied from a poisoned ticket) asks for a write
write_request = "Ticket 'Cannot login to Aura' is fixed: change its status to Closed in the graph."
q_write = llm_sparql([{"role": "system", "content": SYSTEM.replace("a single SPARQL query", "a single SPARQL query or update")},
                      {"role": "user", "content": write_request}])
print("LLM output for a write request:\n", q_write)
print("\nGate verdict:", check_query(q_write)[2])
print("Gate verdict for a hand-written DELETE:", check_query("DELETE WHERE { ?c ex:isSolution true }")[2])
print(f"\nTriples in the store after both attempts: {len(g)} (unchanged)")

LLM output for a write request:
 UPDATE {
  ?ticket ex:hasStatus ex:Closed .
} 
WHERE {
  ?ticket ex:title "Cannot login to Aura" .
}

Gate verdict: ['READ-ONLY GATE: this is a write operation (INSERT/DELETE/UPDATE/...). Only SELECT queries are permitted.']
Gate verdict for a hand-written DELETE: ['READ-ONLY GATE: this is a write operation (INSERT/DELETE/UPDATE/...). Only SELECT queries are permitted.']

Triples in the store after both attempts: 751 (unchanged)


## Run the benchmark: unchecked vs. ontology-checked with repair (Slides 59-60)

In [5]:
def run_rows(q):
    q_full = q if "PREFIX ex:" in q else PFX + q
    return {tuple(short(v) for v in r) for r in g.query(q_full)}

def is_correct(pred_rows, gold):
    if pred_rows is None or len(pred_rows) != len(gold):
        return False
    remaining = list(pred_rows)
    for gold_row in gold:
        hit = next((r for r in remaining if all(any(alt in r for alt in cell) for cell in gold_row)), None)
        if hit is None:
            return False
        remaining.remove(hit)
    return True

MAX_REPAIRS = 3
results, repair_example = [], None
for qid, text, _ in QUESTIONS:
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": text}]
    first = llm_sparql(msgs)

    # (a) Unchecked: execute whatever came back
    try:
        rows_a = run_rows(first)
    except Exception:
        rows_a = None
    correct_a = is_correct(rows_a, GOLD[qid])

    # (b) Checked: deterministic check -> explain violations -> repair (<= 3) -> else "unknown"
    q, repairs, history = first, 0, []
    while True:
        ok, _, violations = check_query(q)
        if ok:
            try:
                rows_b = run_rows(q); break
            except Exception as e:
                violations = [f"EXECUTION: {str(e)[:150]}"]
        history.append((q, violations))
        if repairs == MAX_REPAIRS:
            rows_b = "unknown"; break
        repairs += 1
        msgs += [{"role": "assistant", "content": f"```sparql\n{q}\n```"},
                 {"role": "user", "content": "The query was rejected by the ontology checker:\n- " + "\n- ".join(violations) +
                                             "\nRewrite the query to fix these problems."}]
        q = llm_sparql(msgs)
    correct_b = rows_b != "unknown" and is_correct(rows_b, GOLD[qid])
    if repair_example is None and history and correct_b and not correct_a:
        repair_example = (qid, text, history, q)

    outcome_b = "unknown" if rows_b == "unknown" else ("correct" if correct_b else "wrong")
    results.append({"qid": qid, "a": "correct" if correct_a else "wrong", "b": outcome_b, "repairs": repairs,
                    "violations": [v.split(":")[0] for _, vs in history for v in vs]})
    print(f"{qid}  unchecked={'correct' if correct_a else 'WRONG':<8}  checked={outcome_b:<8}  repairs={repairs}  "
          f"violations={Counter(results[-1]['violations']) or '-'}")

Q01  unchecked=WRONG     checked=correct   repairs=1  violations=Counter({'DIRECTION': 1})


Q02  unchecked=WRONG     checked=correct   repairs=1  violations=Counter({'DIRECTION': 1})


Q03  unchecked=WRONG     checked=correct   repairs=1  violations=Counter({'GROUNDING': 1})


Q04  unchecked=WRONG     checked=wrong     repairs=1  violations=Counter({'DATATYPE': 1})


Q05  unchecked=correct   checked=correct   repairs=0  violations=-


...

Q11  unchecked=WRONG     checked=unknown   repairs=3  violations=Counter({'DOMAIN/RANGE': 7, 'DIRECTION': 2})


Q12  unchecked=correct   checked=correct   repairs=0  violations=-


Q13  unchecked=WRONG     checked=correct   repairs=3  violations=Counter({'DIRECTION': 4, 'DOMAIN/RANGE': 2})


Q14  unchecked=WRONG     checked=wrong     repairs=0  violations=-


Q15  unchecked=WRONG     checked=correct   repairs=1  violations=Counter({'GROUNDING': 1, 'DIRECTION': 1, 'DOMAIN/RANGE': 1})


## One repair, end to end

In [6]:
if repair_example:
    qid, text, history, final_q = repair_example
    print(f"{qid}: {text}\n")
    for i, (q, viol) in enumerate(history):
        print(f"--- attempt {i + 1} (rejected) ---\n{q}")
        for v in viol: print("   ✗", v)
        print()
    print(f"--- final query (passes the checker, correct answer) ---\n{final_q}")
    print("\nAnswer rows:", sorted(run_rows(final_q)))
else:
    print("No question in this run was fixed by a repair.")

Q01: Which tickets were submitted by Marie Dubois? Return the ticket titles.

--- attempt 1 (rejected) ---
SELECT ?ticketTitle WHERE {
  ?ticket ex:submits ?user .
  ?user ex:firstName "Marie" .
  ?user ex:lastName "Dubois" .
  ?ticket ex:title ?ticketTitle .
}
   ✗ DIRECTION: `?ticket ex:submits ?user` is reversed. ex:submits goes from a User to a Ticket; write `?user ex:submits ?ticket`.

--- final query (passes the checker, correct answer) ---
SELECT ?ticketTitle WHERE {
  ?user ex:firstName "Marie" .
  ?user ex:lastName "Dubois" .
  ?user ex:submits ?ticket .
  ?ticket ex:title ?ticketTitle .
}

Answer rows: [('Aura not recognizing department names',), ('Cannot login to Aura',)]


In [7]:
n = len(results)
acc_a = sum(r["a"] == "correct" for r in results) / n
acc_b = sum(r["b"] == "correct" for r in results) / n
unk_b = sum(r["b"] == "unknown" for r in results) / n
err_a = 1 - acc_a
err_b = sum(r["b"] == "wrong" for r in results) / n
viol_counts = Counter(v for r in results for v in r["violations"])
first_try_rejected = sum(r["repairs"] > 0 or r["b"] == "unknown" for r in results)

print("=" * 78)
print("RESULTS SUMMARY — Notebook 7: ontology-checked text-to-SPARQL (15 questions, gpt-4o-mini)")
print("=" * 78)
print(f"{'':32}{'accuracy':>10}{'wrong answers':>16}{'unknown':>10}")
print(f"{'(a) unchecked, terse schema':32}{acc_a:>10.0%}{err_a:>16.0%}{0:>10.0%}")
print(f"{'(b) + ontology check & repair':32}{acc_b:>10.0%}{err_b:>16.0%}{unk_b:>10.0%}")
print(f"\nFirst-attempt queries rejected by the checker: {first_try_rejected}/{n}; "
      f"total repair rounds: {sum(r['repairs'] for r in results)}")
print("Violation types caught:", dict(viol_counts))
print("Write attempts executed against the store: 0 (read-only gate)")
still_wrong = [r["qid"] for r in results if r["b"] == "wrong"]
print(f"Passed every check but still wrong: {still_wrong or 'none'} - structurally valid is not semantically right (Slide 50)")
print("\nLecture message (Slides 59-61): a deterministic ontology check turns silent wrong answers into either")
print("repaired correct answers or an explicit 'unknown' - the LLM writes, the rules decide what may run.")

RESULTS SUMMARY — Notebook 7: ontology-checked text-to-SPARQL (15 questions, gpt-4o-mini)
                                  accuracy   wrong answers   unknown
(a) unchecked, terse schema            20%             80%        0%
(b) + ontology check & repair          67%             27%        7%

First-attempt queries rejected by the checker: 10/15; total repair rounds: 15
Violation types caught: {'DIRECTION': 11, 'GROUNDING': 4, 'DATATYPE': 1, 'DOMAIN/RANGE': 14}
Write attempts executed against the store: 0 (read-only gate)
Passed every check but still wrong: ['Q04', 'Q08', 'Q10', 'Q14'] - structurally valid is not semantically right (Slide 50)

Lecture message (Slides 59-61): a deterministic ontology check turns silent wrong answers into either
repaired correct answers or an explicit 'unknown' - the LLM writes, the rules decide what may run.
